# Compare ALE / Coordinate / DiFuMo Models

Loads the comparison CSV/JSONL written by `experiments/train_ale_cnn.py` and produces the main retrieval and practicality tables.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

COMPARISON_FILE = Path('runs/ale_model_comparison.csv')
if not COMPARISON_FILE.exists():
    raise FileNotFoundError(f'Missing {COMPARISON_FILE}. Run experiments/train_ale_cnn.py first.')
df = pd.read_csv(COMPARISON_FILE)
df.tail()

In [ ]:
# Optional: add rows from existing NeuroVLM/CoordGNN/CoordDeepSet/DiFuMo GAT runs.
# Fill these from their eval JSONs or paper numbers when available.
MANUAL_BASELINES = [
    # {'model_name': 'NeuroVLM MLP', 'preprocessing_type': 'difumo_flatmap', 'uses_difumo': 'yes', 'atlas_free': 'no',
    #  'input_shape': '(28542,)', 'number_of_parameters': None, 'training_time_per_epoch': None, 'peak_vram': None,
    #  'full_recall_curve_auc': 0.81, 'recall@1': None, 'recall@5': None, 'recall@10': None, 'recall@50': None,
    #  'mrr': None, 'median_rank': None, 'random_recall@10': None, 'checkpoint_path': '', 'config_path': ''},
    # dict(model_name='CoordGNN', preprocessing_type='raw_coordinates_graph', uses_difumo='no', atlas_free='yes', ...),
    # dict(model_name='CoordDeepSet', preprocessing_type='raw_coordinates_set', uses_difumo='no', atlas_free='yes', ...),
    # dict(model_name='DiFuMo GAT', preprocessing_type='difumo_coefficients_graph', uses_difumo='yes', atlas_free='no', ...),
]
if MANUAL_BASELINES:
    df = pd.concat([df, pd.DataFrame(MANUAL_BASELINES)], ignore_index=True)
df

In [ ]:
cols = [
    'model_name', 'preprocessing_type', 'uses_difumo', 'atlas_free', 'input_shape',
    'number_of_parameters', 'training_time_per_epoch', 'peak_vram',
    'full_recall_curve_auc', 'recall@1', 'recall@5', 'recall@10', 'recall@50',
    'mrr', 'median_rank', 'random_recall@10', 'checkpoint_path', 'config_path'
]
table = df[cols].sort_values('full_recall_curve_auc', ascending=False)
table

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_df = table.copy()
labels = plot_df['model_name'] + ' / ' + plot_df['preprocessing_type']
ax.barh(labels, plot_df['full_recall_curve_auc'])
ax.invert_yaxis()
ax.set_xlabel('full recall-curve AUC')
ax.set_title('Retrieval AUC')
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for col in ['recall@1', 'recall@5', 'recall@10', 'recall@50']:
    ax.plot(labels, plot_df[col], marker='o', label=col)
ax.tick_params(axis='x', rotation=45)
ax.set_ylabel('recall')
ax.set_title('Fixed top-k retrieval')
ax.legend()
plt.tight_layout()

## Interpretation Template

- If DiFuMo-compatible ALE3DCNN beats the existing MLP, spatial inductive bias helped.
- If DiFuMo-compatible ALE3DCNN does not beat the MLP, the MLP may already be sufficient for that representation.
- If atlas-free ALE3DCNN beats DiFuMo-compatible ALE3DCNN, the atlas/DiFuMo bottleneck may be limiting performance.
- If atlas-free ALEFlatMLP beats atlas-free ALE3DCNN, preprocessing mattered more than CNN architecture.
- If atlas-free ALE3DCNN beats atlas-free ALEFlatMLP, both preprocessing and spatial CNN architecture helped.
- If none beat the MLP, the limiting factor may be coordinate-derived proxy noise, text-pairing noise, or the training objective.